In [ ]:
!pip install datasets==2.19.0

In [ ]:
!pip install seqeval -q

In [ ]:
!pip install transformers huggingface_hub

## Expected Pipeline Flow

Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison


## 1. Dataset Selection

In [1]:
from datasets import load_dataset

dataset = load_dataset("conll2003", trust_remote_code=True)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [3]:
dataset["train"][0] #Understand the dataset format

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [5]:
label_list = dataset["train"].features["chunk_tags"].feature.names
num_labels = len(label_list)
print(label_list)

['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


## 2. Data Preprocessing

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding=True,   
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])  # subwords

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [9]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})


## 3. Model Setup

In [11]:
from transformers import AutoModelForTokenClassification

id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized be

## 4. Training

In [ ]:
%pip install accelerate

In [13]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
  #  evaluation_strategy="epoch",
  #  save_strategy="epoch",
    weight_decay=0.01,
    report_to="none" 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    #compute_metrics=compute_metrics, 
    data_collator=data_collator
)

In [15]:
trainer.train()nn

Step,Training Loss
500,0.482863
1000,0.228060
1500,0.179681
2000,0.159293
2500,0.142052


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2634, training_loss=0.2333754725590201, metrics={'train_runtime': 2903.5586, 'train_samples_per_second': 14.507, 'train_steps_per_second': 0.907, 'total_flos': 3021701491878306.0, 'train_loss': 0.2333754725590201, 'epoch': 3.0})

## 5. Evaluation

In [17]:
import numpy as np
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

predictions, labels, _ = trainer.predict(tokenized_dataset["validation"])
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]

true_labels = [
    [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]

print("Precision:", precision_score(true_labels, true_predictions))
print("Recall:", recall_score(true_labels, true_predictions))
print("F1 Score:", f1_score(true_labels, true_predictions))

print("\nDetailed Report:\n")
print(classification_report(true_labels, true_predictions))

Precision: 0.9184318581763671
Recall: 0.9102742159490973
F1 Score: 0.9143348419225591

Detailed Report:



C:\Users\anya2\anaconda3\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

        ADJP       0.72      0.55      0.62       379
        ADVP       0.77      0.68      0.72       767
       CONJP       0.67      0.20      0.31        10
        INTJ       0.00      0.00      0.00        43
         LST       0.00      0.00      0.00         3
          NP       0.92      0.92      0.92     19363
          PP       0.97      0.97      0.97      4886
         PRT       0.68      0.81      0.74       147
        SBAR       0.83      0.81      0.82       370
          VP       0.91      0.90      0.91      4993

   micro avg       0.92      0.91      0.91     30961
   macro avg       0.65      0.58      0.60     30961
weighted avg       0.92      0.91      0.91     30961



## 6. Inference

In [19]:
from transformers import pipeline

nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

sentence = "John works at Google in California"
result = nlp(sentence)

for r in result:
    print(r["word"], "->", r["entity"])

john -> B-NP
works -> B-VP
at -> B-PP
google -> B-NP
in -> B-PP
california -> B-NP


## 7. Comparison

POS Tagging vs Chunking

| Feature          | POS Tagging                           | Chunking                             |
| ---------------- | ------------------------------------- | ------------------------------------ |
| Level            | Word-level (Grammar)                  | Phrase-level                         |
| Definition       | Assigns grammatical tags to each word | Groups words into meaningful phrases |
| Example          | *John → Noun, runs → Verb*            | *John → Noun Phrase (NP)*            |
| Complexity       | Easy                                  | Medium                               |
| Context Required | Less                                  | More                                 |
| Output           | Tags like NN, VB, JJ                  | Tags like B-NP, I-NP, B-VP           |
| Dependency       | Independent per word                  | Depends on neighboring words         |
| Use Case         | Syntax analysis                       | Information extraction               |

POS tagging focuses on assigning grammatical labels to individual words, whereas chunking groups words into meaningful phrases. Chunking is more complex as it requires contextual understanding of multiple words.

## 8. Report

## - Difference between POS Tagging and Chunking

  POS tagging identifies the grammatical role of each word, such as noun or verb, while chunking groups words into higher-level syntactic units like noun phrases and verb phrases. POS tagging operates at the word level, whereas chunking operates at the phrase level and requires contextual understanding.
  
## - Challenges faced

  Several challenges were encountered during implementation:

- Library version compatibility issues with transformers and related dependencies
- Errors in Trainer API, such as evaluation and callback issues
- Handling variable-length sequences requiring padding
- Aligning labels correctly with subword tokenization
- Class imbalance in the dataset leading to lower performance for rare labels

## - Observations and Insights

  The following observations were made during the project:

- Transformer models like BERT perform very well for sequence labelling tasks
- Frequent classes such as NP and VP achieved high accuracy
- Rare classes had lower performance due to fewer training samples
- Proper preprocessing and label alignment are crucial for model performance
- Token classification tasks require careful handling of special tokens and subwords

Overall, the project demonstrates that transformer-based models are highly effective for NLP tasks such as chunking and POS tagging, providing strong contextual understanding and high accuracy.
